# Smart MCQ Solver

**Three-model pipeline with Weights & Biases tracking:**
- **Model 1 — From Scratch:** MLP trained on TF-IDF features (PyTorch, no pretrained weights)
- **Model 2 — Pretrained Off-the-Shelf:** all-MiniLM-L6-v2 sentence embeddings + cosine similarity ranking
- **Model 3 — Fine-Tuned Pretrained:** LoRA DeBERTa-v3-base with RAG context augmentation

All three models evaluated on the same validation split using MAP@3, Accuracy@1, and F1-macro. Results compared in WandB and a summary table.

# 0. Environment Setup

In [1]:
# Fix version conflicts first
# !pip install -q torchao --upgrade
# !pip install -q peft --upgrade
# !pip install -q faiss-cpu
!pip install -q wandb
# !pip install -q sentencepiece

In [2]:
import os, re, string, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# from sentence_transformers import SentenceTransformer
# from transformers import (
#     AutoTokenizer, AutoModelForMultipleChoice,
#     TrainingArguments, Trainer, EvalPrediction
# )
# from datasets import Dataset
# from peft import get_peft_model, LoraConfig, TaskType
# import faiss
# from sentence_transformers import CrossEncoder

import wandb
from kaggle_secrets import UserSecretsClient

## Configuration

In [3]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX_TO_OPT  = {v: k for k, v in LABEL_MAP.items()}

print(f"Device: {DEVICE}")

# WandB login
secrets       = UserSecretsClient()
wandb_api_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)
print("WandB ready.")

Device: cuda


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


WandB ready.


# 1. Data Preprocessing

In [4]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

train['label'] = train['answer'].map(LABEL_MAP)

## Data Cleaning

Manual inspection of train.csv and test.csv revealed that a large portion of prompts
contain boilerplate noise that adds no semantic value:

- **1806 train / 304 test prompts** start with one of six filler prefixes
  (`"Choose the correct answer:"`, `"Determine the correct option:"`, etc.)
- **1462 train / 228 test prompts** end with one of four filler suffixes
  (`"from the following choices."`, `"carefully."`, etc.)
- Some prompts are wrapped in **double quotes**
- After stripping this noise, **duplicate prompts appear** — different rows
  asking the exact same underlying question with different wrappers

Cleaning before any feature engineering or tokenization ensures:
1. TF-IDF vocabulary isn't polluted by high-frequency boilerplate tokens
2. Embeddings compare actual question content, not wrapper phrasing
3. Duplicate questions don't create artificial signal during training

In [5]:
PREFIXES = [
    "choose the correct answer:",
    "determine the correct option:",
    "identify the correct statement:",
    "pick the best possible answer:",
    "select the most accurate option:",
    "which of the following is correct?",
]
 
SUFFIXES = [
    "from the following choices.",
    "carefully.",
    "based on the given context.",
    "among the listed options.",
]

def clean_prompt(text):
    text = str(text).strip()

    # Remove surrounding double quotes
    if text.startswith('"') and text.endswith('"'):
        text = text[1:-1].strip()

    # Strip filler prefixes (case-insensitive)
    lower = text.lower()
    for prefix in PREFIXES:
        if lower.startswith(prefix):
            text  = text[len(prefix):].strip()
            lower = text.lower()
            break  # only one prefix per prompt

    # Strip filler suffixes (case-insensitive)
    for suffix in SUFFIXES:
        if lower.endswith(suffix):
            text  = text[: -len(suffix)].strip()
            lower = text.lower()
            break  # only one suffix per prompt

    # Normalise internal whitespace
    text = re.sub(r'\s+', ' ', text)

    return text

# Apply to both splits
before_train = len(train)
before_test  = len(test)

train['prompt'] = train['prompt'].map(clean_prompt)
test['prompt']  = test['prompt'].map(clean_prompt)

## Duplication Handling

In [6]:
train["prompt"].nunique()

252

In [7]:
train["prompt"].value_counts().head()

prompt
What is the definition of an improper rotation?                                                                                                                                    21
What is the significance of the high degree of fatty-acyl disorder in the thylakoid membranes of plants?                                                                           20
What was the aim of the Gravity Probe B (GP-B) mission?                                                                                                                            18
What is the gravitomagnetic interaction?                                                                                                                                           18
Which of the following statements accurately describes the impact of Modified Newtonian Dynamics (MOND) on the observed "missing baryonic mass" discrepancy in galaxy clusters?    16
Name: count, dtype: int64

In [8]:
# Report duplicates (keep first occurrence — same question, same options → same answer)
content_cols = ['prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

train_dupes = train.duplicated(subset=content_cols).sum()
train_dupes

np.int64(1380)

*Note:* That's a very unusual dataset, but the numbers are internally consistent.

- 2000 total rows
- 1380 duplicates
- 620 unique MCQs
- 256 unique prompts

In [9]:
train = train.drop_duplicates(subset=content_cols, keep='first').reset_index(drop=True)
 
print(f"Train rows : {before_train} → {len(train)}  ({train_dupes} duplicate MCQs removed)")
print(f"Test rows  : {before_test} → {len(test)}   (no label so duplicates kept)")
print(f"\nSample cleaned prompts:")
print(train['prompt'].head(5).to_string())

Train rows : 2000 → 620  (1380 duplicate MCQs removed)
Test rows  : 500 → 500   (no label so duplicates kept)

Sample cleaned prompts:
0    What is Martin Heidegger's view on the relatio...
1          What is accelerator-based light-ion fusion?
2    What is the term used in astrophysics to descr...
3    What is the concept of simultaneity in Einstei...
4    What is the effect generated by a spinning sup...


## Train/Validation Split

In [10]:
# Fixed 80/20 split — same seed used by all three models for fair comparison
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"train: {train.shape}  |  train_fit: {train_df.shape}  |  val: {val_df.shape}  |  test: {test.shape}")
train.head(3)

train: (620, 9)  |  train_fit: (496, 9)  |  val: (124, 9)  |  test: (500, 7)


,id,prompt,A,B,C,D,E,answer,label
0,1,What is Martin Heidegger's view on the relatio...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,What is the term used in astrophysics to descr...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2


# 2. Shared Evaluation Helpers

All three models are evaluated with the same functions so metrics are directly comparable. Top-1 predicted letter is used as the class label for Accuracy@1 and F1-macro.

In [11]:
# top3_list: list of lists of 3 option letters
def map_at_3(true_answers, top3_list):
    scores = []
    for ans, top3 in zip(true_answers, top3_list):
        top3 = list(top3)
        scores.append(1 / (top3.index(ans) + 1) if ans in top3 else 0)
    return float(np.mean(scores))

def accuracy_at_1(true_answers, top3_list):
    return float(np.mean([t[0] == a for a, t in zip(true_answers, top3_list)]))

def f1_macro(true_answers, top3_list):
    y_true = [LABEL_MAP[a] for a in true_answers]
    y_pred = [LABEL_MAP[t[0]] for t in top3_list]
    return float(f1_score(y_true, y_pred, average='macro', zero_division=0))

# Compute all three metrics and print a summary table row
def evaluate_all(model_name, true_answers, top3_list):
    m3  = map_at_3(true_answers, top3_list)
    a1  = accuracy_at_1(true_answers, top3_list)
    f1  = f1_macro(true_answers, top3_list)
    print(f"  {'Model':<35} {'MAP@3':>8} {'Acc@1':>8} {'F1-macro':>10}")
    print(f"  {model_name:<35} {m3:>8.4f} {a1:>8.4f} {f1:>10.4f}")
    return {"map3": m3, "acc1": a1, "f1": f1}

# Print row-by-row correctness for the first n rows
def correctness_table(val_df, top3_list, n=15):
    check = val_df[['id', 'answer']].copy().head(n)
    check['top3']         = [' '.join(t) for t in top3_list[:n]]
    check['top1_correct'] = [t[0] == a for t, a in zip(top3_list[:n], val_df['answer'][:n])]
    check['in_top3']      = [a in t for t, a in zip(top3_list[:n], val_df['answer'][:n])]
    print(check.to_string(index=False))

# 3. Model 1 — From Scratch: MLP on TF-IDF Features

**Architecture:** TF-IDF vectorizer (no pretrained weights) → MLP with two hidden layers → 5 logits (one per option).

The input is a single TF-IDF vector representing the concatenation of the prompt and all 5 options for a row. The model learns to assign a score to each class (A–E) purely from the bag-of-words feature space.

**No pretrained weights anywhere in this model** — the TF-IDF vocabulary is built from train+test data, and the MLP weights are randomly initialised and trained from scratch.

## TF-IDF features

In [12]:
IF_IDF_MAX_FEATURES=2500

def build_combined(df):
    return df[['prompt'] + OPTION_COLS].astype(str).agg(' '.join, axis=1)

tfidf = TfidfVectorizer(stop_words='english', max_features=IF_IDF_MAX_FEATURES)


# Validation stage
X_train = tfidf.fit_transform(build_combined(train_df)).toarray().astype(np.float32)
X_val   = tfidf.transform(build_combined(val_df)).toarray().astype(np.float32)
y_train = train_df['label'].values.astype(np.int64)
y_val   = val_df['label'].values.astype(np.int64)

# Final training stage
X = tfidf.fit_transform(build_combined(train)).toarray().astype(np.float32)
X_test   = tfidf.transform(build_combined(test)).toarray().astype(np.float32)
y       = train['label'].values.astype(np.int64)

print(f"TF-IDF vocab: {len(tfidf.get_feature_names_out())}  |  X_train: {X_train.shape}  X_val: {X_val.shape}")

TF-IDF vocab: 1000  |  X_train: (496, 1000)  X_val: (124, 1000)


## MLP architecture

In [13]:
INPUT_DIM  = X_train.shape[1]
INPUT_DIM_FULL = X.shape[1]
HIDDEN_DIM = 512
NUM_CLASSES = 5

class MCQ_MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

mlp = MCQ_MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"MLP total parameters: {total_params:,}")
print(mlp)

MLP total parameters: 646,661
MCQ_MLP(
  (net): Sequential(
    (0): Linear(in_features=1000, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=5, bias=True)
  )
)


## DataLoader

In [14]:
BATCH_SIZE = 64

def make_loader(X, y=None, batch_size=32, shuffle=False):
    X = torch.tensor(X, dtype=torch.float32)

    if y is None:
        dataset = TensorDataset(X)
    else:
        y = torch.tensor(y, dtype=torch.long)
        dataset = TensorDataset(X, y)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


# Validation
train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val, y_val, BATCH_SIZE)

# Final
full_train_loader = make_loader(X, y, BATCH_SIZE, shuffle=True)
test_loader       = make_loader(X_test, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

Train batches: 8  Val batches: 2


## Training with WandB

In [15]:
MLP_EPOCHS = 30
MLP_LR     = 3e-4

wandb.init(
    project = "24f3003354-t22026",
    name    = "mlp-scratch",
    config  = {
        "model"       : "MLP from scratch",
        "input_dim"   : INPUT_DIM,
        "hidden_dim"  : HIDDEN_DIM,
        "epochs"      : MLP_EPOCHS,
        "lr"          : MLP_LR,
        "batch_size"  : BATCH_SIZE,
        "dropout"     : 0.3,
        "tfidf_feats" : IF_IDF_MAX_FEATURES,
    }
)

In [16]:
optimizer  = torch.optim.AdamW(mlp.parameters(), lr=MLP_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MLP_EPOCHS)
criterion  = nn.CrossEntropyLoss()

def find_mlp_top3(model, data_loader):
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        for batch in data_loader:
            xb = batch[0].to(DEVICE)
            probs = F.softmax(model(xb), dim=1)
            all_probs.append(probs.cpu())

    probs = torch.cat(all_probs).numpy()
    ranked = np.argsort(-probs, axis=1)
    return [[IDX_TO_OPT[j] for j in row[:3]] for row in ranked]

In [17]:
best_map3 = 0
patience = 5
patience_counter = 0

for epoch in range(1, MLP_EPOCHS + 1):
    # Training
    mlp.train()
    train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    train_loss /= len(train_loader)

    # Evaluation
    top3_val = find_mlp_top3(mlp, val_loader)
    
    # Validation metrics
    val_map3   = map_at_3(val_df['answer'], top3_val)
    val_acc1   = accuracy_at_1(val_df['answer'], top3_val)
    val_f1     = f1_macro(val_df['answer'], top3_val)

    # Wandb Log
    wandb.log({
        "epoch"      : epoch,
        "train_loss" : round(train_loss, 4),
        "val_map3"   : round(val_map3, 4),
        "val_acc1"   : round(val_acc1, 4),
        "val_f1"     : round(val_f1, 4),
    })

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{MLP_EPOCHS} | loss={train_loss:.4f} | MAP@3={val_map3:.4f} | Acc@1={val_acc1:.4f} | F1={val_f1:.4f}")

    # Check and Save best model
    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(
            mlp.state_dict(),
            'best_mlp.pt'
        )
        patience_counter = 0
    else:
        patience_counter += 1

    # Break if score is not improving
    if patience_counter >= patience:
        print(f"Stopping early at epoch {epoch}!")
        break

wandb.finish()
print("\nMLP training complete.")

Epoch  1/30 | loss=1.4064 | MAP@3=0.4691 | Acc@1=0.2903 | F1=0.1970
Epoch  5/30 | loss=0.3109 | MAP@3=0.8172 | Acc@1=0.7581 | F1=0.7668
Epoch 10/30 | loss=0.1050 | MAP@3=0.8575 | Acc@1=0.8226 | F1=0.8151
Stopping early at epoch 12!


epoch,▁▂▂▃▄▄▅▅▆▇▇█
train_loss,█▅▃▃▂▂▂▁▁▁▁▁
val_acc1,▂▁▂▄▇███████
val_f1,▂▁▂▄▇███████
val_map3,▁▁▃▅▇███████
epoch,12
train_loss,0.0802
val_acc1,0.8226
val_f1,0.8151
val_map3,0.8548



MLP training complete.


## MLP final evaluation on validation set

In [18]:
mlp.load_state_dict(torch.load("best_mlp.pt"))
mlp.eval()

mlp_top3_val  = find_mlp_top3(mlp, val_loader)
mlp_metrics   = evaluate_all("Model 1 — MLP from scratch", val_df['answer'], mlp_top3_val)

# print("\nRow-by-row correctness (first 15 rows):")
# correctness_table(val_df, mlp_top3_val)

  Model                                  MAP@3    Acc@1   F1-macro
  Model 1 — MLP from scratch            0.8669   0.8306     0.8298


## Retrain on full data

In [19]:
mlp_full = MCQ_MLP(INPUT_DIM_FULL, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
optimizer_full = torch.optim.AdamW(mlp_full.parameters(), lr=MLP_LR, weight_decay=1e-4)
scheduler_full = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_full, T_max=MLP_EPOCHS)

for epoch in range(1, MLP_EPOCHS + 1):
    mlp_full.train()
    for xb, yb in full_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_full.zero_grad()
        loss = criterion(mlp_full(xb), yb)
        loss.backward()
        optimizer_full.step()
    scheduler_full.step()

print("Full-data retraining complete.")

Full-data retraining complete.


In [20]:
mlp_top3_pred = find_mlp_top3(mlp_full, test_loader)

predictions = [' '.join(row) for row in mlp_top3_pred]
print(predictions[:5])

['A B C', 'B C A', 'B C D', 'E C D', 'C A B']


In [21]:
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': predictions
})

submission.to_csv('submission.csv', index=False)

submission.head()

,ID,Prediction
0,1,A B C
1,2,B C A
2,3,B C D
3,4,E C D
4,5,C A B


In [22]:
raise SystemExit("Stopping notebook execution here.")

SystemExit: Stopping notebook execution here.

# 4. Model 2 — all-MiniLM-L6-v2 (Pretrained)

**Approach:** Encode the prompt and each of the 5 options separately using all-MiniLM-L6-v2. Rank options by cosine similarity to the prompt. No fine-tuning — weights are used exactly as released.

This is the approach from Milestone 2, now evaluated properly on the same val split with all three metrics logged to WandB.

In [ ]:
minilm = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print("MiniLM loaded.")

In [ ]:
# Encode prompt and each option, rank by cosine similarity, return top-3 letters per row
def minilm_top3(df, model, batch_size=64):
    prompt_embs = model.encode(df['prompt'].tolist(), batch_size=batch_size,
                                normalize_embeddings=True, show_progress_bar=True)
    sim_cols = []
    for opt in OPTION_COLS:
        opt_embs = model.encode(df[opt].astype(str).tolist(), batch_size=batch_size,
                                 normalize_embeddings=True, show_progress_bar=False)
        sim_cols.append((prompt_embs * opt_embs).sum(axis=1))

    sim_matrix = np.column_stack(sim_cols)   # (n, 5)
    ranked     = np.argsort(-sim_matrix, axis=1)
    return [[IDX_TO_OPT[j] for j in row[:3]] for row in ranked]

print("Running MiniLM inference on validation set...")
minilm_top3_val = minilm_top3(val_df, minilm)
print("Done.")

## Log to Wand

In [ ]:
mini_metrics = evaluate_all("Model 2 — MiniLM pretrained", val_df['answer'], minilm_top3_val)

wandb.init(
    project = "24f3003354-t22026",
    name    = "minilm-pretrained",
    config  = {
        "model"      : "all-MiniLM-L6-v2",
        "finetuned"  : False,
        "approach"   : "cosine similarity ranking",
        "batch_size" : 64,
    }
)
wandb.log({
    "val_map3" : round(mini_metrics['map3'], 4),
    "val_acc1" : round(mini_metrics['acc1'], 4),
    "val_f1"   : round(mini_metrics['f1'],  4),
})
wandb.finish()

print("\nRow-by-row correctness (first 15 rows):")
correctness_table(val_df, minilm_top3_val)

# 5. Model 3 — Fine-Tuned Pretrained: LoRA DeBERTa-v3-base

**Approach:** RAG-augmented inputs formatted as `Context: [retrieved_doc] [SEP] [prompt] [SEP] [option]` for each of the 5 options, tokenized into `[batch, 5, 192]` tensors, passed through a LoRA-adapted DeBERTa for multiple-choice classification.

Brings together Milestone 3 (RAG retrieval) and Milestone 4 (LoRA fine-tuning + Trainer).

## RAG Pipeline Setup

Builds the knowledge base and FAISS index used by Model 3. Same two-stage retrieval from Milestone 3: bi-encoder (MiniLM) → FAISS top-5 → cross-encoder rerank → single best document injected as context.

In [ ]:
print("Building RAG knowledge base from train correct answers...")
kb = [str(row[row['answer']]) for _, row in train.iterrows()]

kb_embs = minilm.encode(kb, batch_size=64, normalize_embeddings=True,
                          show_progress_bar=True).astype(np.float32)

faiss_index = faiss.IndexFlatIP(kb_embs.shape[1])
faiss_index.add(kb_embs)
print(f"FAISS index ready — {faiss_index.ntotal} vectors.")

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE)
print("Cross-encoder ready.")

In [ ]:
# Two-stage retrieval: FAISS bi-encoder → cross-encoder rerank → best doc
def retrieve_best_doc(prompts, k=5):
    embs = minilm.encode(prompts, batch_size=64, normalize_embeddings=True,
                          show_progress_bar=False).astype(np.float32)
    _, indices = faiss_index.search(embs, k)
    best_docs = []
    for prompt, cand_idx in zip(prompts, indices):
        docs   = [kb[i] for i in cand_idx]
        scores = cross_encoder.predict([[prompt, d] for d in docs])
        best_docs.append(docs[int(np.argmax(scores))])
    return best_docs

print("Retrieving RAG docs for train_fit, val, and test sets...")
train_docs = retrieve_best_doc(train_df['prompt'].tolist())
val_docs   = retrieve_best_doc(val_df['prompt'].tolist())
test_docs  = retrieve_best_doc(test['prompt'].tolist())
print("RAG retrieval complete.")

In [ ]:
DEBERTA_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH    = 192

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
print(f"DeBERTa tokenizer loaded — vocab: {deberta_tokenizer.vocab_size}")

In [ ]:
# Build 5 RAG-augmented (context + prompt + option) strings
def format_rag_choices(prompt, best_doc, row):
    prefix = f"Context: {best_doc} [SEP] {prompt}"
    return [prefix + " [SEP] " + str(row[opt]) for opt in OPTION_COLS]

def make_hf_records(df, best_docs):
    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        choices = format_rag_choices(str(row['prompt']), best_docs[i], row)
        enc = deberta_tokenizer(choices, padding="max_length",
                                truncation=True, max_length=MAX_LENGTH)
        records.append({
            "input_ids"      : enc["input_ids"],
            "attention_mask" : enc["attention_mask"],
            "labels"         : int(LABEL_MAP[row['answer']])
        })
    return records

print("Tokenizing train and val sets for DeBERTa...")
hf_train = Dataset.from_list(make_hf_records(train_df, train_docs)).with_format("torch")
hf_val   = Dataset.from_list(make_hf_records(val_df,   val_docs)).with_format("torch")
print(f"hf_train: {len(hf_train)}  hf_val: {len(hf_val)}")

In [ ]:
base_model  = AutoModelForMultipleChoice.from_pretrained(DEBERTA_MODEL)
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    ranked  = np.argsort(-logits, axis=1)
    top3    = [[IDX_TO_OPT[j] for j in row[:3]] for row in ranked]
    true    = [IDX_TO_OPT[int(l)] for l in labels]
    return {
        "map3" : round(map_at_3(true, top3), 4),
        "acc1" : round(accuracy_at_1(true, top3), 4),
        "f1"   : round(f1_macro(true, top3), 4),
    }

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_deberta_output",
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 2,
    num_train_epochs            = 3,
    learning_rate               = 1e-4,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "map3",
    greater_is_better           = True,
    logging_steps               = 20,
    report_to                   = "wandb",
    run_name                    = "lora-deberta-finetuned",
    bf16                        = torch.cuda.is_available(),
)

trainer = Trainer(
    model           = lora_model,
    args            = training_args,
    train_dataset   = hf_train,
    eval_dataset    = hf_val,
    compute_metrics = compute_metrics,
)

wandb.init(
    project = "24f3003354-t22026",
    name    = "lora-deberta-finetuned",
    config  = {
        "model"        : DEBERTA_MODEL,
        "lora_r"       : 8,
        "lora_alpha"   : 16,
        "max_length"   : MAX_LENGTH,
        "epochs"       : 3,
        "lr"           : 1e-4,
        "batch_size"   : 8,
        "grad_accum"   : 2,
        "rag_k"        : 5,
    }
)

print("Starting DeBERTa LoRA training...")
trainer.train()
wandb.finish()
print("Training complete.")

## DeBERTa final eval

In [ ]:
lora_model.eval()

def deberta_top3(df, best_docs):
    top3_list = []
    for i, (_, row) in enumerate(df.iterrows()):
        choices = format_rag_choices(str(row['prompt']), best_docs[i], row)
        enc = deberta_tokenizer(choices, padding="max_length", truncation=True,
                                max_length=MAX_LENGTH, return_tensors="pt")
        inp = {k: v.unsqueeze(0).to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = lora_model(**inp).logits[0]
        probs  = F.softmax(logits, dim=-1).cpu().numpy()
        ranked = np.argsort(-probs)
        top3_list.append([IDX_TO_OPT[j] for j in ranked[:3]])
    return top3_list

print("Running DeBERTa inference on validation set...")
deberta_top3_val = deberta_top3(val_df, val_docs)
deberta_metrics  = evaluate_all("Model 3 — LoRA DeBERTa fine-tuned", val_df['answer'], deberta_top3_val)

print("\nRow-by-row correctness (first 15 rows):")
correctness_table(val_df, deberta_top3_val)

# 7. Model Comparison

All three models evaluated on the same validation split (same 20% held-out rows, same random seed). Summary table + WandB comparison run.

In [ ]:
results = {
    "Model 1 — MLP (from scratch)"       : mlp_metrics,
    "Model 2 — MiniLM (pretrained)"      : mini_metrics,
    "Model 3 — LoRA DeBERTa (fine-tuned)": deberta_metrics,
}

print("=" * 65)
print(f"{'Model':<42} {'MAP@3':>7} {'Acc@1':>7} {'F1':>7}")
print("=" * 65)
for name, m in results.items():
    print(f"{name:<42} {m['map3']:>7.4f} {m['acc1']:>7.4f} {m['f1']:>7.4f}")
print("=" * 65)

## WandB comparison run

In [ ]:
# ── WandB comparison run ─────────────────────────────────────────────
wandb.init(
    project = "24f3003354-t22026",
    name    = "model-comparison",
    config  = {"split": "80/20 val", "random_state": 42}
)

for name, m in results.items():
    wandb.log({
        f"{name}/map3" : m['map3'],
        f"{name}/acc1" : m['acc1'],
        f"{name}/f1"   : m['f1'],
    })

# Log as a WandB Table for easy side-by-side view in the dashboard
comparison_table = wandb.Table(
    columns = ["Model", "MAP@3", "Acc@1", "F1-macro"],
    data    = [[name, m['map3'], m['acc1'], m['f1']] for name, m in results.items()]
)
wandb.log({"comparison_table": comparison_table})
wandb.finish()
print("Comparison run logged to WandB.")

# 8. Final Submission

Refit DeBERTa on the **full** train.csv (no val holdout) for maximum training signal, then generate test predictions with the content-word length tiebreaker blended in for uncertain cases.

In [ ]:
# Content-word length tiebreaker (from V3 — 0.48 standalone score)
def content_word_len(text):
    text = str(text).lower().translate(str.maketrans('', '', string.punctuation))
    return len([w for w in text.split() if w not in ENGLISH_STOP_WORDS])

def length_score_row(row):
    lengths = np.array([content_word_len(str(row[opt])) for opt in OPTION_COLS], dtype=float)
    span = lengths.max() - lengths.min()
    return (lengths - lengths.min()) / span if span > 0 else np.full(5, 0.5)

BLEND_THRESHOLD = 0.05
LENGTH_WEIGHT   = 0.30

def blend(model_probs, len_scores):
    top2 = np.sort(model_probs)[::-1][:2]
    if top2[0] - top2[1] < BLEND_THRESHOLD:
        return (1 - LENGTH_WEIGHT) * model_probs + LENGTH_WEIGHT * len_scores
    return model_probs

In [ ]:
# Refit on full train — reuse already-retrieved test_docs
print("Tokenizing full train for final DeBERTa refit...")
hf_train_full = Dataset.from_list(
    make_hf_records(train.reset_index(drop=True),
                    retrieve_best_doc(train['prompt'].tolist()))
).with_format("torch")

base_model_final  = AutoModelForMultipleChoice.from_pretrained(DEBERTA_MODEL)
lora_model_final  = get_peft_model(base_model_final, LoraConfig(
    r=8, lora_alpha=16, target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS
))

final_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_deberta_final",
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 2,
    num_train_epochs            = 3,
    learning_rate               = 1e-4,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    logging_steps               = 20,
    report_to                   = "none",
    bf16                        = torch.cuda.is_available(),
    save_strategy               = "no",
)

Trainer(model=lora_model_final, args=final_args,
        train_dataset=hf_train_full).train()
print("Final model trained on full train.csv.")

## Generate test predictions

In [ ]:
lora_model_final.eval()
predictions = []

print(f"Running inference on {len(test)} test rows...")
for i, (_, row) in enumerate(test.iterrows()):
    choices = format_rag_choices(str(row['prompt']), test_docs[i], row)
    enc = deberta_tokenizer(choices, padding="max_length", truncation=True,
                            max_length=MAX_LENGTH, return_tensors="pt")
    inp = {k: v.unsqueeze(0).to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = lora_model_final(**inp).logits[0]
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    probs  = blend(probs, length_score_row(row))
    ranked = np.argsort(-probs)
    predictions.append(' '.join([IDX_TO_OPT[j] for j in ranked[:3]]))
    if i % 100 == 0:
        print(f"  {i}/{len(test)} done...")

print("Inference complete.")

In [ ]:
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})

assert len(submission) == len(test)
assert submission['Prediction'].str.split().apply(len).eq(3).all()
valid = set(OPTION_COLS)
assert submission['Prediction'].str.split().apply(lambda x: set(x).issubset(valid)).all()

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"submission.csv saved — {submission.shape}")
submission.head(10)